In [14]:
# ! sudo apt install mafft
# ! pip install ete3

In [3]:
import subprocess
import time
import sys
from pathlib import Path
from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

In [17]:
Entrez.email = "aminaimanalieva03@gmail.com"

# accession-номера
accessions = ["L11288", "X04025", "AF173605", "X82564", "AF173636"]

outdir = Path(".")
raw_fasta = outdir / "18S_regions_minimum.fasta"           # исходные
aligned_fasta = outdir / "18S_regions_minimum_aligned.fasta" # выравненные
clean_fasta = outdir / "geneX_clean_softtrim.fasta"        # после фильтрации
tree_file = outdir / "geneX_tree.newick"                   # файл с деревсм

def fetch_record(acc):
    with Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text") as h:
        return SeqIO.read(h, "genbank")

# 18S рРНК
def extract_18S_or_full(rec):
    for f in rec.features:
        if f.type.lower() == "rrna":
            prods = f.qualifiers.get("product", [])
            for p in prods:
                if "18S" in p or "18 s" in p.lower():
                    return SeqRecord(f.extract(rec.seq), id=rec.id, description=rec.description)
    # если 18S не нашли то сохраняем полную последовательность
    return SeqRecord(rec.seq, id=rec.id, description=rec.description)

records = []
for acc in accessions:
    r = fetch_record(acc)
    records.append(extract_18S_or_full(r))
    time.sleep(0.3)

SeqIO.write(records, raw_fasta, "fasta")

# Сначала попробовала MAFFT, если не получилось, можно MUSCLE
mafft_cmd = ["mafft", "--auto", str(raw_fasta)]
muscle_cmd = ["muscle", "-in", str(raw_fasta), "-out", str(aligned_fasta)]

proc = subprocess.run(mafft_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
if proc.returncode == 0 and proc.stdout.strip():
    aligned_fasta.write_text(proc.stdout)
else:
    proc2 = subprocess.run(muscle_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if proc2.returncode != 0:
        # если ниче не вышло то ошибка
        sys.exit(3)

aln = AlignIO.read(str(aligned_fasta), "fasta")
nseq = len(aln)                        # число последовательностей
L = aln.get_alignment_length()         # длина выравнивания


GAP_THRESHOLD = 0.5   # допускается не более 50% пропусков в колонке
N_THRESHOLD = 0.2     # допускается не более 20% неопределённых оснований (N)

cols_keep = [
    i for i in range(L)
    if (aln[:, i].count("-")/nseq) <= GAP_THRESHOLD and
       (aln[:, i].upper().count("N")/nseq) <= N_THRESHOLD
]
# здесь мы уже выбрали хорошие позиции

new_recs = []
for rec in aln:
    seq_str = "".join(rec.seq[i] for i in cols_keep)
    new_recs.append(SeqRecord(Seq(seq_str), id=rec.id, description=rec.description))

AlignIO.write(MultipleSeqAlignment(new_recs), str(clean_fasta), "fasta")

# строим дерево теперь
aln2 = AlignIO.read(str(clean_fasta), "fasta")

# матрица расстояний (модель 'identity' - доля совпадений)
calc = DistanceCalculator('identity')
dm = calc.get_distance(aln2)

constructor = DistanceTreeConstructor()
tree = constructor.nj(dm)

# идем по латимерии и вверх
tips = [t.name for t in tree.get_terminals()]
for t in tips:
    if t.startswith("L11288"):
        tree.root_with_outgroup(t)
        break

Phylo.write(tree, str(tree_file), "newick")


1

In [16]:
from ete3 import Tree
from Bio import SeqIO

t = Tree("geneX_tree.newick", format=1)

name_map = {}
for record in SeqIO.parse("/content/geneX_clean_softtrim.fasta", "fasta"):
    parts = record.description.split(None, 1)
    nice_name = parts[1] if len(parts) > 1 else record.id
    name_map[record.id] = nice_name

for leaf in t.get_leaves():
    if leaf.name in name_map:
        leaf.name = name_map[leaf.name] + " " + leaf.name

print(t.get_ascii(show_internal=False))



            /-Ciconia nigra 18S ribosomal RNA gene, complete sequence AF173636.1
         /-|
      /-|   \-Alligator mississippiensis 18S ribosomal RNA gene, complete sequence AF173605.1
     |  |
   /-|   \-M.musculus 45S pre rRNA gene X82564.1
  |  |
--|   \-Xenopus laevis 18S ribosomal RNA X04025.1
  |
   \-Latimeria chalumnae 18S ribosomal RNA L11288.1


Птица (*Ciconia nigra*) и крокодил *(Alligator mississippiensis*)  →  *Archosauria*

Млекопитающее *(Mus musculus*) уходит в отдельную ветвь →
Земноводное (*Xenopus laevis*) отходит раньше →
Latimeria (*Latimeria chalumnae*)